# Finding CpG islands with a hidden Markov model

*bioinformatics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/bioinformatics/hmm_cpg_islands)

A DNA sequence switches unseen between ordinary background and GC-rich CpG islands. A two-state hidden Markov model finds them: forward–backward gives the probability of being in an island at every position, Viterbi the single best segmentation, and Baum–Welch learns the model from the sequences alone. Each step along the sequence is a batched matrix product over the states, kept in log space. Watch the posterior lock onto the planted islands.

**Vector form**

$$\boldsymbol\alpha_t = \mathbf e(x_t) \odot \big(A^{\mathsf T}\boldsymbol\alpha_{t-1}\big),\qquad \boldsymbol\beta_t = A\,\big(\mathbf e(x_{t+1}) \odot \boldsymbol\beta_{t+1}\big),\qquad P(z_t = k \mid x) = \frac{\alpha_{tk}\,\beta_{tk}}{P(x)}$$

**Index (tensor) form**

$$e_{btk} = x_{bt\alpha}E_{k\alpha},\qquad \alpha_{btj} = e_{btj}\,\alpha_{b,t-1,i}A_{ij},\qquad \beta_{bti} = A_{ij}\,e_{b,t+1,j}\,\beta_{b,t+1,j},\qquad \gamma_{btk} = \frac{\alpha_{btk}\beta_{btk}}{\alpha_{bTj}\mathbf 1_j}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{bt\alpha}$ | `one_hot(x_codes)` | (B, T, 4) | letter at position $t$ of sequence $b$, one-hot over A, C, G, T |
| $\pi_k,\ A_{ij},\ E_{k\alpha}$ | `pi, A, E = initial_guess(device)` | (2,), (2, 2), (2, 4) | start, transition and emission probabilities; state 0 background, state 1 island |
| $\log e_{btk} = x_{bt\alpha}\log E_{k\alpha}$ | `einsum('bta,ka->btk', x, E.log())` | (B, T, K) | repeated $\alpha$ ⇒ summed: how likely each state is to emit the letter seen |
| $\alpha_{btj} = e_{btj}\,\alpha_{b,t-1,i}A_{ij}$ | `forward_step(log_alpha, A, log_e)  # einsum('bi,ij->bj', ...)` | (B, K) | repeated $i$ ⇒ summed; stored as $\log\alpha$ with the largest entry pulled out (log-sum-exp) so nothing underflows |
| $\beta_{bti} = A_{ij}e_{b,t+1,j}\beta_{b,t+1,j}$ | `backward_step(log_beta, A, log_e_next)  # einsum('ij,bj->bi', ...)` | (B, K) | repeated $j$ ⇒ summed: probability of the rest of the sequence given state $i$ |
| $\gamma_{btk}$ | `softmax(log_alpha + log_beta, dim=-1)` | (B, T, K) | posterior $P(z_t = k \mid x)$; $\gamma_{bt1}$ is the blue curve |
| $\delta_{btj} = \log e_{btj} + \max_i(\delta_{b,t-1,i} + \log A_{ij})$ | `viterbi_step(delta, A, log_e)` | (B, K) | Viterbi: the same recursion with max in place of sum, plus back-pointers for the best path |
| $\xi_{ij} = \alpha_{bti}A_{ij}e_{b,t+1,j}\beta_{b,t+1,j}/P(x_b)$ | `einsum('bt,bti,ij,btj->ij', ...)` | (K, K) | repeated $b, t$ ⇒ summed: expected number of $i \to j$ switches; Baum–Welch sets $A_{ij} \propto \xi_{ij}$ |

The data are synthetic: every sequence has islands of 100–300 letters planted at random places, with more C and G inside. One letter says almost nothing about the state, but the chain $A$ makes states persist, so the forward and backward passes pool the evidence of whole stretches. Baum–Welch is expectation–maximisation: compute $\gamma$ and $\xi$ under the current model, then set $E_{k\alpha}$ to the expected letter frequencies in each state and $A_{ij}$ to the expected switch frequencies; the likelihood never decreases. Starting from a vague guess it ends close to the true model (green line). Real CpG islands are rich in the dinucleotide CG rather than just in C and G; Durbin et al. model that with 8 states (A+, C+, G+, T+, A−, …) whose transitions are the dinucleotide frequencies. Lower the island GC content to make the problem harder.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/bioinformatics/hmm_cpg_islands/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors
from .equation import log_emissions


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n_iter = 3 if fast_mode() else params["iterations"]
    truth = tensors.true_model(params["gc_island"], dev)  # (π, A, E) that generated the data
    x_codes, z = tensors.planted_sequences(params["n_seq"], params["length"], params["n_islands"], truth[2], 0)
    x = tensors.one_hot(x_codes)  # (B, T, 4)
    B, T = z.shape
    print(f"{B} sequences × {T} letters, {float(z.float().mean()):.1%} of positions inside planted islands")
    for k, name in enumerate(tensors.STATES):
        gc = x[..., 1:3].sum(-1)[z == k].mean()
        print(f"  GC content in {name:10s}: {float(gc):.3f}")

    pi, A, E = tensors.initial_guess(dev)
    models, log_lik, post0 = solve.baum_welch(x, pi, A, E, n_iter)
    for it in sorted({0, 1, 2, n_iter}):
        print(f"Baum–Welch iteration {it:2d}: log P(x) = {log_lik[it]:.1f}")
    pi, A, E = models[-1]
    print("learned emissions  A     C     G     T      (true)")
    for k, name in enumerate(tensors.STATES):
        row = "  ".join(f"{p:.3f}" for p in E[k].tolist())
        true_row = " ".join(f"{p:.2f}" for p in truth[2][k].tolist())
        print(f"  {name:10s}  {row}   ({true_row})")
    print(f"learned mean island length 1/(1 − A_11) = {1 / (1 - float(A[1, 1])):.0f} letters")

    _, _, log_px_true = solve.e_step(x, *truth)
    print(f"log P(x) under the true model: {float(log_px_true.sum()):.1f}")
    path = solve.viterbi(log_emissions(x, E), pi, A)  # (B, T)
    gamma, _, _ = solve.e_step(x, pi, A, E)
    acc_v = float((path == z).double().mean())
    acc_p = float(((gamma[..., 1] > 0.5).long() == z).double().mean())
    print(f"accuracy against the planted islands: Viterbi {acc_v:.1%}, posterior > 0.5 {acc_p:.1%}")

    paths0 = [solve.viterbi(log_emissions(x[:1], m[2]), m[0], m[1])[0] for m in models]  # sequence 0, every model
    return plot.animate(
        x_codes[0], z[0], post0, paths0, models, truth, log_lik, float(log_px_true.sum()), out_dir / "cpg_islands.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/bioinformatics/hmm_cpg_islands/equation.py
r"""Hidden Markov model → tensor translation.

A DNA sequence x_1 … x_T is emitted by a hidden state z_t ∈ {background, island} that follows a Markov chain:
P(z_1 = k) = π_k, P(z_{t+1} = j | z_t = i) = A_{ij}, P(x_t = α | z_t = k) = E_{kα}. CpG islands are GC-rich, so
the island state emits C and G more often. With x_{btα} one-hot (sequence b, position t, letter α):

    e_{btk} = x_{btα} E_{kα}                                       (α summed: emission probability of every state)
    α_{btj} = e_{btj} \sum_i α_{b,t-1,i} A_{ij}                    forward  (i summed)
    β_{bti} = \sum_j A_{ij} e_{b,t+1,j} β_{b,t+1,j}                backward (j summed)
    P(x_b) = \sum_k α_{bTk},       γ_{btk} = α_{btk} β_{btk} / P(x_b)        posterior P(z_t = k | x)

α underflows after a few hundred letters, so it is stored as log α. Each step is still the matrix product above:
pull out the largest entry m, multiply exp(log α − m) by A with einsum, and add m back (the log-sum-exp trick).

Viterbi replaces the sum by a max:  δ_{btj} = log e_{btj} + max_i (δ_{b,t-1,i} + log A_{ij}).
Baum–Welch re-estimates π, A, E from the expected counts γ and
ξ_{ij} = \sum_{bt} α_{bti} A_{ij} e_{b,t+1,j} β_{b,t+1,j} / P(x_b)   (b, t summed).
"""

import torch


def log_emissions(x: torch.Tensor, E: torch.Tensor) -> torch.Tensor:
    r"""log e_{btk} = x_{btα} log E_{kα}   (α summed).   (B, T, 4), (K, 4) → (B, T, K)"""
    return torch.einsum("bta,ka->btk", x, E.log())


def forward_step(log_alpha: torch.Tensor, A: torch.Tensor, log_e: torch.Tensor) -> torch.Tensor:
    r"""log α_{btj} = log e_{btj} + log \sum_i α_{b,t-1,i} A_{ij}   (i summed).   (B, K), (K, K), (B, K) → (B, K)"""
    m = log_alpha.amax(dim=1, keepdim=True)  # (B, 1) shift that keeps exp() in range
    return log_e + m + torch.einsum("bi,ij->bj", (log_alpha - m).exp(), A).log()


def backward_step(log_beta: torch.Tensor, A: torch.Tensor, log_e_next: torch.Tensor) -> torch.Tensor:
    r"""log β_{bti} = log \sum_j A_{ij} e_{b,t+1,j} β_{b,t+1,j}   (j summed).   (B, K) → (B, K)"""
    v = log_e_next + log_beta  # (B, K)
    m = v.amax(dim=1, keepdim=True)
    return m + torch.einsum("ij,bj->bi", A, (v - m).exp()).log()


def viterbi_step(delta: torch.Tensor, A: torch.Tensor, log_e: torch.Tensor):
    r"""δ_{btj} = log e_{btj} + max_i (δ_{b,t-1,i} + log A_{ij}); also returns the best previous state i.  (B, K)"""
    scores = delta[:, :, None] + A.log()[None]  # (B, K, K): come from i (dim 1), go to j (dim 2)
    return log_e + scores.amax(dim=1), scores.argmax(dim=1)


def posterior(log_alpha: torch.Tensor, log_beta: torch.Tensor) -> torch.Tensor:
    r"""γ_{btk} = α_{btk} β_{btk} / \sum_k α_{btk} β_{btk}.   (B, T, K) → (B, T, K), sums to 1 over k"""
    return torch.softmax(log_alpha + log_beta, dim=-1)


def expected_transitions(log_alpha, log_beta, log_e, A, log_px) -> torch.Tensor:
    r"""ξ_{ij} = \sum_{bt} α_{bti} A_{ij} e_{b,t+1,j} β_{b,t+1,j} / P(x_b)   (b, t summed).   → (K, K)

    Every factor is rescaled by its largest entry per (b, t) so the exponentials stay in range; the scales come back
    through the weight w_{bt}.
    """
    la = log_alpha[:, :-1]  # (B, T-1, K): α at t
    lv = (log_e + log_beta)[:, 1:]  # (B, T-1, K): e β at t+1
    s, r = la.amax(-1, keepdim=True), lv.amax(-1, keepdim=True)
    w = (s + r)[..., 0] - log_px[:, None]  # (B, T-1)
    return torch.einsum("bt,bti,ij,btj->ij", w.exp(), (la - s).exp(), A, (lv - r).exp())


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/bioinformatics/hmm_cpg_islands/tensors.py
"""Build the concrete tensors: the HMM parameters and synthetic DNA with planted CpG islands (seeded)."""

import torch
import torch.nn.functional as F

ALPHABET = "ACGT"
STATES = ("background", "island")


def model(stay_bg: float, stay_island: float, bg: list, island: list, device: torch.device):
    """Two-state HMM (π, A, E): start (2,), transitions (2, 2), emissions (2, 4) over A, C, G, T."""
    pi = torch.tensor([0.9, 0.1], dtype=torch.float64)  # float64: log-likelihoods of long sequences are large
    A = torch.tensor([[stay_bg, 1 - stay_bg], [1 - stay_island, stay_island]], dtype=torch.float64)
    E = torch.tensor([bg, island], dtype=torch.float64)
    E = E / E.sum(dim=1, keepdim=True)
    return pi.to(device), A.to(device), E.to(device)


def true_model(gc_island: float, device: torch.device):
    """Model that generates the data: background 40 % GC, islands `gc_island` GC, islands ~200 letters long."""
    at = (1 - gc_island) / 2
    return model(0.998, 0.995, [0.3, 0.2, 0.2, 0.3], [at, gc_island / 2, gc_island / 2, at], device)


def initial_guess(device: torch.device):
    """A vague starting point for Baum–Welch: the island state is only slightly GC-richer and switches too often."""
    return model(0.98, 0.95, [0.27, 0.23, 0.23, 0.27], [0.22, 0.28, 0.28, 0.22], device)


def planted_sequences(n_seq: int, length: int, n_islands: int, E: torch.Tensor, seed: int):
    """Hidden states z (B, T) with `n_islands` islands of 100–300 letters planted at random places, and letters
    x (B, T) drawn from the emission row of each position's state."""
    gen = torch.Generator().manual_seed(seed)
    z = torch.zeros(n_seq, length, dtype=torch.long)
    for b in range(n_seq):
        for _ in range(n_islands):
            size = int(torch.randint(100, 301, (1,), generator=gen))
            start = int(torch.randint(0, length - size, (1,), generator=gen))
            z[b, start : start + size] = 1
    probs = E.cpu()[z]  # (B, T, 4): emission probabilities at every position
    x = torch.multinomial(probs.reshape(-1, 4), 1, generator=gen).reshape(n_seq, length)
    return x.to(E.device), z.to(E.device)


def one_hot(x: torch.Tensor) -> torch.Tensor:
    """x_{btα}: (B, T) letter codes → (B, T, 4)."""
    return F.one_hot(x, 4).double()


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/bioinformatics/hmm_cpg_islands/solve.py
"""Forward–backward, Viterbi and Baum–Welch: loops over the positions t, every sequence of the batch at once."""

import torch

from problems._common.progress import steps

from .equation import (
    backward_step,
    expected_transitions,
    forward_step,
    log_emissions,
    posterior,
    viterbi_step,
)


def forward(log_e: torch.Tensor, pi: torch.Tensor, A: torch.Tensor):
    """log α of shape (B, T, K) and the log-likelihood log P(x_b) of every sequence (B,)."""
    log_alpha = torch.empty_like(log_e)
    log_alpha[:, 0] = pi.log() + log_e[:, 0]
    for t in range(1, log_e.shape[1]):
        log_alpha[:, t] = forward_step(log_alpha[:, t - 1], A, log_e[:, t])
    return log_alpha, torch.logsumexp(log_alpha[:, -1], dim=-1)


def backward(log_e: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
    """log β of shape (B, T, K), with β_T = 1."""
    log_beta = torch.zeros_like(log_e)
    for t in range(log_e.shape[1] - 2, -1, -1):
        log_beta[:, t] = backward_step(log_beta[:, t + 1], A, log_e[:, t + 1])
    return log_beta


def viterbi(log_e: torch.Tensor, pi: torch.Tensor, A: torch.Tensor) -> torch.Tensor:
    """Most probable state path of every sequence: (B, T) state indices."""
    B, T, K = log_e.shape
    delta = pi.log() + log_e[:, 0]
    prev = torch.zeros(B, T, K, dtype=torch.long, device=log_e.device)  # back-pointers
    for t in range(1, T):
        delta, prev[:, t] = viterbi_step(delta, A, log_e[:, t])
    path = torch.empty(B, T, dtype=torch.long, device=log_e.device)
    path[:, -1] = delta.argmax(dim=1)
    for t in range(T - 1, 0, -1):  # follow the back-pointers from the end
        path[:, t - 1] = prev[:, t].gather(1, path[:, t : t + 1])[:, 0]
    return path


def e_step(x: torch.Tensor, pi, A, E):
    """Posteriors γ (B, T, K), expected transitions ξ (K, K) and log-likelihoods (B,) under one model."""
    log_e = log_emissions(x, E)
    log_alpha, log_px = forward(log_e, pi, A)
    log_beta = backward(log_e, A)
    gamma = posterior(log_alpha, log_beta)
    xi = expected_transitions(log_alpha, log_beta, log_e, A, log_px)
    return gamma, xi, log_px


def m_step(x: torch.Tensor, gamma: torch.Tensor, xi: torch.Tensor):
    """New π, A, E from expected counts: π_k = <γ_{b1k}>, A ∝ ξ, E_{kα} ∝ γ_{btk} x_{btα} (b, t summed)."""
    pi = gamma[:, 0].mean(dim=0)
    A = xi / xi.sum(dim=1, keepdim=True)
    counts = torch.einsum("btk,bta->ka", gamma, x)
    return pi, A, counts / counts.sum(dim=1, keepdim=True)


def baum_welch(x: torch.Tensor, pi, A, E, n_iter: int):
    """EM for n_iter rounds. Returns the list of models (n_iter + 1 of them, the first is the starting guess),
    the total log-likelihood before each update, and the posterior P(island) of sequence 0 under every model."""
    models, log_lik, post0 = [(pi, A, E)], [], []
    for _ in steps(n_iter, "Baum–Welch"):
        gamma, xi, log_px = e_step(x, pi, A, E)
        log_lik.append(float(log_px.sum()))
        post0.append(gamma[0, :, 1])
        pi, A, E = m_step(x, gamma, xi)
        models.append((pi, A, E))
    gamma, _, log_px = e_step(x, pi, A, E)
    log_lik.append(float(log_px.sum()))
    post0.append(gamma[0, :, 1])
    return models, log_lik, torch.stack(post0)


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/bioinformatics/hmm_cpg_islands/plot.py
"""Animate Baum–Welch: posterior P(island) along sequence 0 against the planted truth, one frame per iteration."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

from problems._common.plotting import frame_indices, save_gif

STRIP = ListedColormap(["#ffffff", "#f59e0b"])


def animate(x0, z0, post0, paths0, models, truth, log_lik, true_ll, out: Path) -> Path:
    """x0, z0 (T,) letters and true states of sequence 0; post0 (I+1, T); paths0 list of (T,) Viterbi paths;
    models list of (π, A, E); truth (π, A, E); log_lik list (I+1,); true_ll float."""
    T = len(z0)
    pos = np.arange(T)
    truth_z = z0.cpu().numpy()
    gc = x0.cpu().numpy()
    gc = ((gc == 1) | (gc == 2)).astype(float)
    window = 51
    gc_smooth = np.convolve(gc, np.ones(window) / window, mode="same")  # GC fraction in a sliding window
    post = post0.cpu().numpy()
    paths = [p.cpu().numpy() for p in paths0]
    E_true = truth[2].cpu().numpy()
    n = len(models)

    fig = plt.figure(figsize=(9, 5))
    grid = fig.add_gridspec(2, 2, height_ratios=[1.25, 1])
    ax = fig.add_subplot(grid[0, :])
    ax_l = fig.add_subplot(grid[1, 0])
    ax_e = fig.add_subplot(grid[1, 1])

    ax.fill_between(pos, 0, truth_z, step="mid", color="#dbeafe", label="planted island")
    ax.plot(pos, gc_smooth, color="#9ca3af", lw=0.8, label=f"GC fraction ({window}-letter window)")
    (line,) = ax.plot(pos, post[0], color="#2563eb", lw=1.3, label="posterior P(island)")
    strip = ax.imshow(paths[0][None], cmap=STRIP, vmin=0, vmax=1, aspect="auto", extent=(-0.5, T - 0.5, -0.2, -0.07))
    ax.text(T * 0.005, -0.135, "Viterbi", fontsize=7, va="center", color="#92400e")
    ax.set_xlim(0, T - 1)
    ax.set_ylim(-0.22, 1.3)
    ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
    ax.set_xlabel("position $t$ in sequence 0")
    ax.legend(loc="upper right", fontsize=7, ncol=3, framealpha=0.9)
    title = ax.set_title(" ", fontsize=10)  # placeholder so tight_layout leaves room

    its = np.arange(n)
    ax_l.plot(its, log_lik, color="#9ca3af", lw=1, marker="o", ms=3)
    ax_l.axhline(true_ll, color="#16a34a", ls="--", lw=1, label="true model")
    (dot,) = ax_l.plot([], [], "o", color="#2563eb", ms=7)
    ax_l.set_xlabel("Baum–Welch iteration")
    ax_l.set_ylabel("log P(x)")
    ax_l.legend(fontsize=7, loc="lower right")
    ax_l.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

    letters = np.arange(4)
    bars = []
    for k, (color, name) in enumerate([("#9ca3af", "background"), ("#2563eb", "island")]):
        bars.append(ax_e.bar(letters + (k - 0.5) * 0.38, models[0][2][k].cpu().numpy(), 0.36, color=color, label=name))
        ax_e.plot(letters + (k - 0.5) * 0.38, E_true[k], "_", color="#111827", ms=12, mew=2)
    ax_e.set_xticks(letters)
    ax_e.set_xticklabels(list("ACGT"))
    ax_e.set_ylim(0, 0.5)
    ax_e.set_ylabel("emission $E_{k\\alpha}$")
    ax_e.set_title("bars: current model, black ticks: true", fontsize=9)
    ax_e.legend(fontsize=7, loc="upper left", ncol=2)

    def update(i: int):
        line.set_ydata(post[i])
        strip.set_data(paths[i][None])
        acc = (paths[i] == truth_z).mean()
        title.set_text(f"iteration {i}: Viterbi agrees with the planted islands at {acc:.1%} of positions")
        dot.set_data([i], [log_lik[i]])
        for k in range(2):
            for bar, h in zip(bars[k], models[i][2][k].tolist()):
                bar.set_height(h)

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(n, 60), out, fps=2)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_seq': 8,
    'length': 2000,
    'n_islands': 2,
    'gc_island': 0.65,
    'iterations': 15
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.bioinformatics.hmm_cpg_islands")]:
    del sys.modules[name]
main = importlib.import_module("problems.bioinformatics.hmm_cpg_islands.main")
problem_dir = Path("problems/bioinformatics/hmm_cpg_islands")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
